# Utility Operations Intelligence — Demo

Run the cells top to bottom. Each step explains what it does and what changes as a result, so you can follow the workflow whether you're presenting it or building it yourself.

The workflow: generate operational data + 50 documents, expose them through 3 Genie spaces + 4 metric views + a Knowledge Assistant, orchestrate them with a supervisor agent, and serve it all through a branded Databricks App.

| Step | Action | How |
|---|---|---|
| 1 | Generate 13 source tables (≈11.5M rows) | scripted |
| 2 | Build 2 rollups + 4 metric views | scripted |
| 3 | Create 3 Genie spaces | scripted |
| 4 | Upload 50 PDFs to a UC Volume | scripted |
| 5a | Build the Knowledge Assistant | **UI (Agent Bricks)** — prompts provided below |
| 5b | Build the Supervisor Multi-Agent System | **UI (Agent Bricks)** — prompt provided below |
| 6 | Grant the app service principal, then deploy the app | scripted |
| 7 | The narrative & example questions to ask the app | read + present |

All settings live in the **`config`** notebook — the single place to configure this demo.

## Setup

Install dependencies (once), then load the shared configuration and helpers.

In [0]:
%pip install polars numpy mimesis pandas 'databricks-connect>=16.4,<17.0'

In [0]:
dbutils.library.restartPython()

In [0]:
%run ./config

In [0]:
describe()

Before running anything, open the **`config`** notebook and confirm the values match your environment. Every step reads from it — if you change one, re-run the Setup cells above.

In [0]:
ensure_schema_and_volume()

---
## Step 1: Generate source data

Creates the 13 source tables (`regions`, `sites`, `assets`, `telemetry` ≈10.5M rows, `outages`, `work_orders`, `asset_financials`, ...) in `{CATALOG}.{SCHEMA}`. **≈15-25 min.**

In [0]:
%run ./1-data/generate

### Step 1b: Sanity SQL
Read-only checks that the causal chains (anomalies -> outages -> emergency work orders, overdue maintenance -> outages) are intact.

In [0]:
%run ./1-data/sanity_sql

---
## Step 2: Build metric views

Creates `asset_daily_summary` + `region_daily_summary` rollups, then the 4 Unity Catalog metric views.

In [0]:
%run ./2-metric-views/build_metric_views

### Step 2b: Validate every measure
Expected: 52 OK, 3 expected zeros, 0 errors.

In [0]:
%run ./2-metric-views/test_metric_views

---
## Step 3: Create the 3 Genie spaces

Builds the serialized space payloads, POSTs them, and captures the resulting space IDs into `GENIE_SPACE_IDS` (used by the supervisor in Step 5b and the grants in Step 6).

In [0]:
%run ./3-genie-spaces/build_genie_spaces

In [0]:
import json
w = workspace_client()
_created = {}
for name, path in [
    ('Grid Operations', '/tmp/grid_create.json'),
    ('Financial Performance', '/tmp/fin_create.json'),
    ('Maintenance & Workforce', '/tmp/maint_create.json'),
]:
    payload = json.loads(open(path).read())
    try:
        resp = w.api_client.do('POST', '/api/2.0/genie/spaces', body=payload)
        _created[name] = resp.get('space_id')
        print(f"OK  {name}: {resp.get('space_id')}")
    except Exception as e:
        print(f'skip {name} (may already exist): {e}')

if _created:
    GENIE_SPACE_IDS = _created
    print('\nGENIE_SPACE_IDS captured. Paste this into config.ipynb for reproducible re-runs:')
    print('GENIE_SPACE_IDS =', json.dumps(GENIE_SPACE_IDS, indent=4))


---
## Step 4: Upload the 50 PDFs to the UC Volume

The PDFs in `4-documents/pdfs/` are pre-generated. This refreshes the narrative anchors, then uploads the PDFs to `/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}/` — the Knowledge Assistant's source.

In [0]:
%run ./4-documents/anchor_narrative

In [0]:
import os
from pathlib import Path
pdf_dir = Path(demo_root()) / '4-documents' / 'pdfs'
target = f'/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}'
pdfs = sorted(pdf_dir.glob('*.pdf'))
print(f'Uploading {len(pdfs)} PDFs -> {target}')
for pdf in pdfs:
    dbutils.fs.cp(f'file:{pdf}', f'{target}/{pdf.name}')
print(f'Uploaded {len(pdfs)} PDFs')


---
## Step 5a: Build the Knowledge Assistant  *(UI — Agent Bricks)*

This step can't be scripted — build it in **AI/ML -> Agents -> Create -> Knowledge Assistant**. Use the exact values below.

**Knowledge source:** Unity Catalog Volume -> `{catalog}` / `{schema}` / `reports` (run the cell below to print your exact path).

**Answer model:** Claude Haiku 4.5 &nbsp;·&nbsp; **top_k:** 5-8 &nbsp;·&nbsp; **Reranker:** off

**Description field — paste this:**

```
Operations and maintenance documents for Cascadia Renewable Operations, a renewable energy utility, covering calendar year 2025. Contains 50 reports across nine types: Outage Investigation Reports (OIR), Corrective Action Reports (CAR), Emergency Maintenance Reports (EMR), Root Cause Analyses (RCA), Vendor Repair Agreements (VRA), Field Maintenance Summaries (FMS), Repair Approval Memos (RAM), Vendor Technical Manuals (VTM), and Operational Escalation Reports (OER). Use this knowledge for questions about specific reports by ID (e.g. RCA-2025-001, CAR-2025-118), root-cause narratives, outage investigations, vendor conduct and pricing (Apex Emergency Grid Solutions, Cascadia Power Services, Siemens Gamesa, Vestas), emergency procurement, repair-vs-replacement disputes, corrective-action backlog, and vibration/overheating failure analysis on specific assets (e.g. WIN-10130, TRA-10561, NW-PowerPool-WIND-024).
```

**Optional >> Instructions field — paste this:**

```
You answer questions using only the retrieved utility operations and maintenance documents.

- Lead with the direct answer in 1-2 sentences, then support it with specifics from the documents.
- Always cite the source document ID inline (e.g. "per RCA-2025-001" or "see CAR-2025-118").
- When the question asks what should be done, report the remediations, corrective actions, repair-vs-replacement recommendations, and approvals as stated in the documents (e.g. Corrective Action Reports, Repair Approval Memos, pending change requests like MOC-2025-022). Attribute each to its source; do not invent recommendations of your own.
- When documents disagree - especially an OEM Vendor Technical Manual vs. an internal Corrective Action or standard (e.g. Siemens Gamesa vibration threshold vs. Cascadia Standard MS-2025-03) - surface the conflict explicitly rather than picking a side.
- Quote exact figures, dates, thresholds, vendor names, and asset IDs as written.
- Do not use outside knowledge. If the documents don't contain the answer, say so.
```

When it deploys, note the endpoint name (e.g. `ka-XXXX-endpoint`) and smoke-test with *"What does RCA-2025-001 say?"* — it should return text with a citation.

In [0]:
print('Knowledge source volume:')
print(f'  Catalog : {CATALOG}')
print(f'  Schema  : {SCHEMA}')
print(f'  Volume  : {VOLUME_NAME}')
print(f'  Path    : /Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}/  (50 PDFs)')


---
## Step 5b: Build the Supervisor Multi-Agent System  *(UI — Agent Bricks)*

In **AI/ML -> Agents -> Create -> Multi-Agent System**:

- **Name:** `Utility Operations Supervisor`
- **Supervisor model:** Claude Haiku 4.5
- **Allow parallel tool calls:** ON &nbsp;·&nbsp; **Streaming:** ON &nbsp;·&nbsp; **Max iterations:** 6
- **Endpoint scale-to-zero:** OFF (avoids cold starts during demos)

**System prompt — paste this:**

```
You are the Operations Intelligence Assistant for Cascadia Renewable Operations, a renewable energy utility. You orchestrate four expert tools, route each question to the right one(s), and synthesize a single clear answer from what they return.

DOMAIN CONTEXT (for routing and vocabulary only - never state these as facts unless a tool returns them):
- The company operates across 8 grid regions. NW-PowerPool (the "western region") and CAISO carry the most operational and cost problems.
- Field service vendors: Cascadia Power Services (CPS) is the preferred preventive/scheduled partner; Apex Emergency Grid Solutions is the high-cost emergency contractor. Equipment OEMs (Siemens Gamesa, Vestas, GE, Nordex, ABB, Siemens Energy) supply manuals and warranty service.
- Recurring themes: emergency vendor cost escalation, repeat-failure "problem assets," preventive-maintenance deferral, and repair-vs-replacement disputes between OEM manuals and internal standards.

TOOLS AND ROUTING:
- Grid Operations Genie - operational metrics: forced outages, lost MWh, capacity factor, MTBF/MTTR, telemetry, anomalies, reliability. Route here for "how many," "which asset/region," outage and generation questions.
- Financial Performance Genie - revenue, OpEx, vendor invoice/emergency-repair cost, margin, market price exposure. Route here for cost, spend, and financial-impact questions.
- Maintenance & Workforce Genie - work orders, technicians, preventive vs. reactive work, schedule adherence, safety. Route here for maintenance-activity and workforce questions.
- Knowledge Assistant - the document corpus (OIR, CAR, EMR, RCA, VRA, VTM, RAM, OER, FMS). Route here for any question naming a report ID, root-cause narratives, vendor conduct, procurement policy, corrective actions/recommendations, or repair-vs-replacement disputes.

ROUTING RULES:
- Call multiple tools IN PARALLEL when a question spans structured metrics and document context. Do not wait for one before starting the next.
- "Why are emergency vendor costs rising and what should we do?" is multi-tool: use Financial (cost trend), Grid Operations (the outages driving it), AND the Knowledge Assistant (vendor conduct + documented corrective actions).
- Questions naming a report ID or "what does <document> say" go to the Knowledge Assistant only.
- Prefer the MEASURE-based results the Genie spaces return; do not recompute numbers yourself.

ANSWER STYLE:
- Lead with the direct answer in 1-2 sentences, then a short bullet list or small table of specifics.
- Cite document IDs inline when the Knowledge Assistant supplies them (e.g. "see RCA-2025-001").
- When sources disagree (e.g. an OEM manual vs. an internal standard), surface the conflict.
- For "what should we do" questions, report the remediations the documents state; attribute each to its source and do not invent recommendations.
- Never speculate. If a tool returns no data, say so plainly.
```

**Add 4 tools** — the 3 Genie spaces (run the cell below for your live space IDs) + the Knowledge Assistant endpoint from Step 5a:

| Display name | Type | Target |
|---|---|---|
| Grid Operations | Genie Space | *Grid Operations id (below)* |
| Financial Performance | Genie Space | *Financial Performance id (below)* |
| Maintenance & Workforce | Genie Space | *Maintenance & Workforce id (below)* |
| Utility Operations Knowledge | Endpoint | *your `ka-*-endpoint`* |

**Save & Deploy.** Note the resulting `mas-*-endpoint` name.

In [0]:
print('Genie space IDs to add as tools:')
for label, sid in GENIE_SPACE_IDS.items():
    print(f'  {label:24s} {sid}')


### Capture the endpoint names from Step 5
Paste the `mas-*` and `ka-*` endpoint names from the two UI builds into **`config`** (`SUPERVISOR_ENDPOINT` and `KA_ENDPOINTS`), then re-run `%run ./config`. The cell below just confirms they're set before Step 6.

In [0]:
assert SUPERVISOR_ENDPOINT, 'Set SUPERVISOR_ENDPOINT in config (from Step 5b), then re-run %run ./config'
assert KA_ENDPOINTS, 'Set KA_ENDPOINTS in config (from Step 5a), then re-run %run ./config'
print('Supervisor endpoint:', SUPERVISOR_ENDPOINT)
print('KA endpoints       :', KA_ENDPOINTS)


---
## Step 6: Grant the app service principal, then deploy

This is the step that makes the app actually work. `deploy_app` **creates** the app (minting its service principal), **grants** that SP access to the supervisor + KA endpoints, the 3 Genie spaces, the warehouse, and the catalog/schema/volume, and only **then deploys** — so the app is live *and* authorized in one shot. No more runtime 403s.

Everything it needs comes from `config` (endpoints, `GENIE_SPACE_IDS`, warehouse, catalog/schema).

In [0]:
%run ./6-app/deploy_app

---
## Step 7: The narrative & questions to ask the app

The demo is deployed. This final step is the payoff — the story the AI can uncover, and the questions to ask the app to reveal it. Open the app URL printed in Step 6 and work down the list.

### The story the AI can discover

**Cascadia Renewable Operations** closed FY2025 with headline revenue of **$857M** — but its western footprint is quietly deteriorating. Two clusters of forced outages — the **Spring Vibration Cascade** (May 2025) and the **Fall Overheating Sequence** (Oct-Nov 2025) — together lost **≈27,400 MWh** and triggered **$14.4M** in emergency vendor invoices. The same few assets keep failing; the same few vendors keep getting called.

The discovery: **NW-PowerPool and CAISO are subsidizing operational neglect through emergency vendor contracts that reward outages instead of preventing them.**

- **Apex Emergency Grid Solutions** bills ≈**4x** Cascadia Power Services for the same work and consistently recommends full replacement over repair.
- **Siemens Gamesa**'s OEM manual mandates replacement at vibration thresholds ≈30% lower than the internal Cascadia Standard **MS-2025-03**; the override (**MOC-2025-022**) has been "pending engineering review" for 9+ months.
- **NW-PowerPool-WIND-024** has 28+ forced outages but no replacement plan — Apex has been invoiced for emergency repairs on the same two turbines 11 times.

*(This is the essence of the former `4-documents/NARRATIVE_BIBLE.md`, surfaced here as the closing step. The full bible remains in `4-documents/` as the source of truth for regenerating documents.)*

### Example questions to ask the app

Grouped by the tool each one exercises. The *expected answer* is a presenter cue — it's what the data supports, so you can confirm the agent routed and answered correctly.

**Key Question**
- *Why are emergency vendor costs rising in NW-PowerPool, and what should we do about it?*

**Grid Operations (Genie — operational metrics)**
- *Which region had the most forced outages this year?* → ERCOT (433)
- *Which single asset lost the most generation, and how much?* → WIN-10130, ≈20,494 MWh across 14 outages
- *Show forced outages by region and month.*

**Financial Performance (Genie — cost & revenue)**
- *How much did we spend on emergency vendor invoices in NW-PowerPool?* → ≈$8.9M
- *Compare Apex Emergency Grid Solutions' rates to Cascadia Power Services.* → Apex ≈4x higher
- *What was total FY2025 revenue?* → ≈$857M

**Maintenance & Workforce (Genie — work orders & crews)**
- *Which assets had repeat emergency repairs by the same vendor?* → e.g. TRA-10561 (twice, 6 months apart)
- *How does preventive vs. reactive work break down by region?*

**Knowledge Assistant (RAG over the 50 PDFs)**
- *What does RCA-2025-001 say?*
- *What does RCA-2025-003 say about WIN-10130?*
- *What does the Siemens Gamesa manual say about vibration thresholds, and how does it compare to MS-2025-03?*

**Hybrid (the supervisor fans out to multiple tools — the headline demo moment)**
- *Why are emergency vendor costs rising in NW-PowerPool, and what should we do about it?*
- *Why does WIN-10130 keep failing, and what's the documented recommended fix?*

Watch the routing pills (blue = Genie, green = Knowledge Assistant) and confirm answers cite document IDs. The hybrid questions are where the multi-agent value lands.

---
To rebrand for a customer: replace `6-app/static/logo.png` and edit the CSS variables in `6-app/static/style.css`, then re-run Step 6.